# The decoder: how does a score matrix become links, and why the same way for every row?

Most rows of Table 1 end in interface (I): for every reuse word a list of `(source_index, probability)` with `-1` as the null. The step from those rows to edges is a design decision of its own in the alignment literature, so the paper fixes one default (average both directions where the model has two, threshold, one-to-one assignment) and keeps the alternatives as an ablation block. This notebook runs every decoder of `retexo/baselines/decoder.py` on the same hand-made rows and shows what each one keeps.

> **No model.** The rows are the `FWD` and `REV` rows of `tests/test_e29.py`: three reuse words over three source words with one honest disagreement between the directions.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, ".")

import json
import pandas as pd
from pathlib import Path

from retexo.baselines.decoder import BaselineDecoder

FWD = [sorted([(0, 0.8), (1, 0.1), (-1, 0.1)], key=lambda x: -x[1]),
       sorted([(1, 0.45), (2, 0.35), (-1, 0.2)], key=lambda x: -x[1]),
       sorted([(2, 0.6), (-1, 0.4)], key=lambda x: -x[1])]
REV = [sorted([(0, 0.9), (-1, 0.1)], key=lambda x: -x[1]),
       sorted([(1, 0.5), (-1, 0.5)], key=lambda x: -x[1]),
       sorted([(1, 0.7), (2, 0.2), (-1, 0.1)], key=lambda x: -x[1])]

def matrix(rows, n_source=3, label="s"):
    return pd.DataFrame([{**{f"{label}{s}": dict(row).get(s, 0.0) for s in range(n_source)}, "null": dict(row).get(-1, 0.0)}
                         for row in rows], index=[f"t{t}" for t in range(len(rows))])

print("forward rows p(s | t)"); display(matrix(FWD))
print("reverse rows p(t | s)"); display(matrix(REV, label="t").rename(index=lambda x: x.replace("t", "s")))


forward rows p(s | t)


,s0,s1,s2,null
t0,0.8,0.10,0.00,0.1
t1,0.0,0.45,0.35,0.2
t2,0.0,0.00,0.60,0.4


reverse rows p(t | s)


,t0,t1,t2,null
s0,0.9,0.0,0.0,0.1
s1,0.0,0.5,0.0,0.5
s2,0.0,0.7,0.2,0.1


## The bidirectional average

Nagata et al. (2020) average the two directions cell by cell; a missing entry counts as zero and the null keeps the forward null. Reuse word `t1` prefers `s1` in the forward direction (0.45) but `s2`'s reverse row points at `t1` with 0.7, so after averaging `t1` prefers `s2` (0.525 against 0.475).


In [2]:
avg = BaselineDecoder.symmetrise_average(FWD, REV, 3)
matrix(avg)


,s0,s1,s2,null
t0,0.85,0.050,0.000,0.1
t1,0.00,0.475,0.525,0.2
t2,0.00,0.000,0.400,0.4


## Every decoder on the same rows

The default thresholds the averaged rows at theta and assigns one-to-one by the Hungarian method with a private null column per word. The alternatives are the rows of the Table 2 decoder block.


In [3]:
rows = []
for name in ("default", "argmax", "mutual", "threshold", "intersect", "gdf", "raw"):
    for theta in (0.3, 0.6):
        links, extra = BaselineDecoder.decode(name, FWD, theta=theta, rev_rows=REV, n_source=3)
        rows.append({"decoder": name, "theta": theta, "links": links, "extra edges": extra})
pd.DataFrame(rows)


,decoder,theta,links,extra edges
0,default,0.3,"[0, 2, -1]",[]
1,default,0.6,"[0, -1, -1]",[]
2,argmax,0.3,"[0, 2, -1]",[]
3,argmax,0.6,"[0, 2, -1]",[]
4,mutual,0.3,"[0, 1, -1]",[]
5,mutual,0.6,"[0, 1, -1]",[]
6,threshold,0.3,"[0, 2, 2]",[]
7,threshold,0.6,"[0, -1, -1]",[]
8,intersect,0.3,"[0, 1, 2]","[(1, 2)]"
9,intersect,0.6,"[0, 1, 2]","[(1, 2)]"


Reading the table: at theta 0.3 the default gives `[0, 2, -1]` because the assignment `t1 -> s2, t2 -> null` (0.525 + 0.4) beats `t1 -> s1, t2 -> s2` (0.475 + 0.4); at theta 0.6 only `t0 -> s0` survives. Mutual argmax keeps `t1 -> s1` (each other's best) and drops `t2 -> s2` because `s2`'s best is `t1`. Grow-diag-final starts from the intersection of the two argmax alignments and grows into the union.


## Tuning the null threshold on a dev fold

The driver picks theta on the dev fold by token accuracy (the headline metric), ties going to the larger theta. On three copies of the toy pair whose gold is `[0, -1, -1]`, the plateau from 0.55 upwards is the answer.


In [4]:
gold = [[0, -1, -1]] * 3
theta = BaselineDecoder.tune_null_threshold([FWD] * 3, gold, rev_per_pair=[REV] * 3, n_source_per_pair=[3] * 3)
print("theta:", theta, "->", BaselineDecoder.decode_default(FWD, theta=theta, rev_rows=REV))
grid = [(t, BaselineDecoder.decode_default(FWD, theta=t, rev_rows=REV)) for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7)]
pd.DataFrame(grid, columns=["theta", "links"])


theta: 0.85 -> [0, -1, -1]


,theta,links
0,0.2,"[0, 2, -1]"
1,0.3,"[0, 2, -1]"
2,0.4,"[0, 2, -1]"
3,0.5,"[0, 2, -1]"
4,0.6,"[0, -1, -1]"
5,0.7,"[0, -1, -1]"


## The full system's stack

The champion decodes with an identity bonus in logit space (E9), a lemma re-ranking of the top-k (E23), a null scale (E18) and a bonus on the cells a fine-tuned rater proposes; with every dial at its neutral value the stack is the default decoder.


In [5]:
from retexo.baselines.record import Record, Edge
record = Record(id="d/1", level="gold", fold=4, source_work="", source_tokens=["arma", "b", "c"], reuse_work="",
                reuse_tokens=["arma", "y", "z"], pair_label="cit", links=[Edge(0, 0, "COPY")])
print("neutral stack:", BaselineDecoder.decode_stack(FWD, record, w=0.0, k=1.0, theta=0.3, rev_rows=REV))
print("identity bonus 2.0 on arma:", matrix(BaselineDecoder.identity_bonus(FWD, record.source_tokens, record.reuse_tokens, 2.0)).iloc[0].round(3).to_dict())


neutral stack: [0, 2, -1]
identity bonus 2.0 on arma: {'s0': 0.967, 's1': 0.016, 's2': 0.0, 'null': 0.016}


## The pair-level functions and their corpus-level originals

The decoders here are pair-level re-statements of the corpus-level originals in `retexo/aligners/assignment.py` (`AssignmentPolicy.links_hungarian`, `.links_argmax`, `Reranker.rerank`, `.with_null_scale`, ported from the E8, E9 and E18 runners now under `attic/scripts/`); the unit tests pin each to its original on these rows, which is a stronger tie than an import.


In [6]:
from retexo.aligners.assignment import AssignmentPolicy
print("hungarian equals the original links_hungarian:", BaselineDecoder.hungarian(FWD) == AssignmentPolicy.links_hungarian([FWD])[0])
print("raw equals the original links_argmax:", BaselineDecoder.decode_raw(FWD) == AssignmentPolicy.links_argmax([FWD])[0])


hungarian equals the original links_hungarian: True
raw equals the original links_argmax: True


## Limits worth knowing

- The Hungarian objective sums probabilities from different softmaxes; the E8 result says it helps anyway, but the objection stands.
- Grow-diag-final and the intersection are many-to-many; only the primary link per reuse word enters the per-token lists, the rest travel as extra edges.
- Theta is tuned per method and fold on token accuracy; a paper table that leads with link F1 would tune on that instead (an open question of note 31).
- The stack's lemma re-ranking needs the record's cached lemmas and the vectors; on an external set it is a no-op.
